# Data investigator for ny taxi dataset

In [9]:
from pyarrow import parquet
import os
import polars as pl

In [51]:
parquet_file = "sample_data/yellow_tripdata_2026-05.parquet"
df = pl.read_parquet(parquet_file)
metadata = file.metadata
print(metadata.schema.names)
print(sum(1 for _ in df.iter_rows()))

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
3977805


### Calculate trip duration

In [ ]:
df = df.with_columns([(df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).alias("tpep_duration")])

In [17]:
print(df["tpep_duration"].head)
print(df["tpep_dropoff_datetime"].head)

<bound method Series.head of shape: (3_977_805,)
Series: 'tpep_duration' [duration[μs]]
[
	28m 21s
	15m 21s
	5m 29s
	39m 22s
	12m 28s
	…
	29m 15s
	12m 56s
	6m 38s
	10m 58s
	26m 17s
]>
<bound method Series.head of shape: (3_977_805,)
Series: 'tpep_dropoff_datetime' [datetime[μs]]
[
	2026-05-16 22:59:54
	2026-05-05 06:02:35
	2026-05-19 07:51:05
	2026-05-01 23:56:06
	2026-05-01 23:13:19
	…
	2026-05-31 00:47:40
	2026-05-06 23:05:34
	2026-05-25 10:36:27
	2026-05-27 20:18:14
	2026-05-07 17:27:54
]>


### Calculate hours with greater demand 

In [48]:
df = df.with_columns([pl.col("tpep_dropoff_datetime").dt.truncate("1h").alias("tpep_dropoff_time"), pl.col("tpep_pickup_datetime").dt.truncate("1h").alias("tpep_pickup_time")])
dropoff_pick = (
    df.group_by(
        pl.col("tpep_dropoff_time")
    )
    .agg(pl.len().alias("dropoff_in_hour"))
    .sort("tpep_dropoff_time")
)

df = df.join(dropoff_pick, how='left', on='tpep_dropoff_time')


pickup_pick = (
    df.group_by(
        pl.col("tpep_pickup_time")
    )
    .agg(pl.len().alias("pickup_in_hour"))
    .sort("tpep_pickup_time")
)

df = df.join(pickup_pick, how='left', on = 'tpep_pickup_time')

print(df.select(["dropoff_in_hour", "tpep_dropoff_time", "pickup_in_hour", "tpep_pickup_time"]).head)

<bound method DataFrame.head of shape: (3_977_805, 4)
┌─────────────────┬─────────────────────┬────────────────┬─────────────────────┐
│ dropoff_in_hour ┆ tpep_dropoff_time   ┆ pickup_in_hour ┆ tpep_pickup_time    │
│ ---             ┆ ---                 ┆ ---            ┆ ---                 │
│ u32             ┆ datetime[μs]        ┆ u32            ┆ datetime[μs]        │
╞═════════════════╪═════════════════════╪════════════════╪═════════════════════╡
│ 9198            ┆ 2026-05-16 22:00:00 ┆ 9692           ┆ 2026-05-16 22:00:00 │
│ 1793            ┆ 2026-05-05 06:00:00 ┆ 964            ┆ 2026-05-05 05:00:00 │
│ 4475            ┆ 2026-05-19 07:00:00 ┆ 5472           ┆ 2026-05-19 07:00:00 │
│ 8340            ┆ 2026-05-01 23:00:00 ┆ 8325           ┆ 2026-05-01 23:00:00 │
│ 8340            ┆ 2026-05-01 23:00:00 ┆ 8325           ┆ 2026-05-01 23:00:00 │
│ …               ┆ …                   ┆ …              ┆ …                   │
│ 8361            ┆ 2026-05-31 00:00:00 ┆ 7788         

### Calculate areas with greater demand

In [53]:
pickup_location = (
    df.group_by(
        pl.col("PULocationID")
    )
    .agg(pl.len().alias("pickups_in_location"))
    .sort("PULocationID")
)

df = df.join(pickup_location, how='left', on='PULocationID')

dropoff_location = (
    df.group_by(
        pl.col("DOLocationID")
    )
    .agg(pl.len().alias("dropoffs_in_location"))
    .sort("DOLocationID")
)

df = df.join(dropoff_location, how='left', on='DOLocationID')

In [54]:
print(df.select(["DOLocationID", "dropoffs_in_location", "PULocationID", "pickups_in_location"]).head)

<bound method DataFrame.head of shape: (3_977_805, 4)
┌──────────────┬──────────────────────┬──────────────┬─────────────────────┐
│ DOLocationID ┆ dropoffs_in_location ┆ PULocationID ┆ pickups_in_location │
│ ---          ┆ ---                  ┆ ---          ┆ ---                 │
│ i32          ┆ u32                  ┆ i32          ┆ u32                 │
╞══════════════╪══════════════════════╪══════════════╪═════════════════════╡
│ 209          ┆ 11595                ┆ 230          ┆ 109730              │
│ 107          ┆ 71547                ┆ 42           ┆ 14369               │
│ 263          ┆ 80293                ┆ 140          ┆ 76156               │
│ 144          ┆ 38182                ┆ 236          ┆ 172739              │
│ 234          ┆ 89147                ┆ 114          ┆ 57818               │
│ …            ┆ …                    ┆ …            ┆ …                   │
│ 16           ┆ 924                  ┆ 33           ┆ 5422                │
│ 238          ┆ 86301